# MACHINE LEARNING PROJECT

### Track T4 – Semi-Supervised Learning (SSL)
The objective of the project is to study and apply semi-supervised learning techniques to tabular data, analyzing how the limited availability of labels affects the performance of predictive models.

I'm interested in cybersecurity, so I chose the CSE-CIC-IDS2018 Intrusion CSVs (IDS 2018) dataset for the classification problem.

The dataset is based on logs from university servers, which recorded various DoS attacks during the publicly available period.
In total, there are eighty columns within this dataset, each of which corresponds to an entry in the IDS logging system that the Unversity of New Brunswick has in place.

Each entry in the dataset is originally labeled with multiple classes, such as Benign, FTP-BruteForce, and Other. To simplify the task and formulate it as a binary classification problem, I will consolidate all non-Benign labels into a single Malicious category. This way, the model will learn to distinguish between Benign and Malicious network traffic.

## Import libraries and models
I import the libraries and modules that we will need for the project.

In [ ]:
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
import os

## Load and manipulate the dataset
I load the dataset using only a single large CSV instead of all the files.This simplifies processing, avoids mismatched type issues, and reduces memory usage.  
At the same time, I convert all labels that are not "Benign" into a single "Malicious" category.  
I use Dask that is a library which performs lazy loading to avoid loading the entire dataset into memory.

In [ ]:
def showPlot(width, height, title, xDescription, yDescription, values):
    plt.figure(figsize=(width, height))
    plt.bar(values.index, values.values, color=['green', 'red'])
    plt.title(title)
    plt.xlabel(xDescription)
    plt.ylabel(yDescription)
    plt.show()

dataset = dd.read_csv('Dataset/02-20-2018.csv', assume_missing = True)
dataset['Label'] = dataset['Label'].map(lambda x: 'Malicious' if x != 'Benign' else 'Benign')
counts = dataset["Label"].value_counts(dropna=False).compute()
print(counts,"\n")
showPlot(6, 4, "Distribution of the Classes", "Class", "Count", counts)

As I can see, all entries are labeled, but the dataset is strongly imbalanced toward benign samples.  
Now I check the entire dataset for entries with any missing feature values.

In [ ]:
def printStats(showPlotFlag):
    dirty_entries_dt = dataset[dataset.isna().any(axis=1)]
    dirty_entries = dirty_entries_dt.shape[0].compute()
    total_entries = dataset.shape[0].compute()
    print(f"Entries that have at least 1 null feature value: {dirty_entries}\nTotal Entries: {total_entries}\nPercentage of dirty entries: {dirty_entries / total_entries:.2%}\n")
    if(dirty_entries > 0):
        print("Analyzing dirty entries:\n" , dirty_entries_dt["Label"].value_counts().compute(), "\n")
    if(showPlotFlag):
        counts = pd.Series([total_entries, dirty_entries], index = ["Total Entries", "Dirty Entries"])
        showPlot(6, 4, "Total Entries vs Dirty Entries", "Type", "Count", counts)
    
printStats(True)

At this point, we see that entries with missing values are less than 1% of the total dataset. Dropping features or filling missing values doesn’t make sense, since most of features are important in traffic analysis, and assigning values could introduce noise and break the real patterns. Therefore, I will remove these incomplete entries from the dataset. Moreover, as noted above, the dataset is heavily imbalanced towards Benign entries (there are many more of them), so removing the dirty entries, which are all Benign, is even more justified.

In [ ]:
dataset = dataset.dropna()
printStats(False)

According to a statistical study conducted by Cloudflare, approximately 6.8% of Internet traffic comes from malicious activities. The dataset shows a percentage of malicious activity relative to benign activity of about 7.8%, so the data reflects real-world conditions. I could apply oversampling techniques, i.e., increasing the number of malicious entries, for example by duplicating them, but this may lead to overfitting. Alternatively, I could use undersampling techniques, reducing the number of benign entries at the cost of losing important information for the model. Since the goal of the project is to demonstrate the effectiveness of a semi-supervised learning approach, I will avoid balancing the dataset for now, but I will apply a model training technique that gives higher weight to malicious instances so that the model learns to predict both classes well, not just the benign ones.

## Features Cleaning
This dataset contains network traffic analysis data between devices and university servers, so it is necessary to carefully examine all the features available in each entry and remove those that are not relevant for analyzing future network traffic and making predictions, thereby essentially avoiding model overfitting.

In [ ]:
print(dataset.columns, "\n\nNumber of features: ", dataset.shape[1])

Among all these features, we can remove the <span style="color:red"><b>Flow ID</b></span>, which is unique and therefore carries no weight in the learning process. The <span style="color:red"><b>destination IP</b></span> is not relevant because it only indicates which university server the packets are sent to and does not help distinguish malicious from benign traffic. Similarly, the <span style="color:red"><b>source IP</b></span> is not useful, as it could be spoofed and, like the destination IP, does not aid in discrimination; in fact, it can cause overfitting—if the model learns that a specific IP is benign, it will continue to trust it blindly, which is very dangerous. The <span style="color:red"><b>source port</b></span> is also not important, since it is often random and, being frequently unknown, analyzing it usually makes little sense. On the other hand, the destination port could be useful because malicious attacks often target well-known ports, such as those for DNS, email services, or Sony services, so analyzing it can contribute to traffic analysis and model learning. The <span style="color:red"><b>timestamp</b></span> could be useful in a time series study, but in our case, we remove it because it does not provide any relevant information.

In [ ]:
dataset = dataset.drop(["Flow ID", "Src IP", "Src Port", "Dst IP", "Timestamp"], axis=1)
print(dataset.columns, "\n\nNumber of features: ", dataset.shape[1])

The dataset indicates that the most important attributes are <span style="color:green"><b>Dst Port</b></span>, <span style="color:green"><b>Protocol</b></span>, <span style="color:green"><b>Flow Duration</b></span>, <span style="color:green"><b>Tot Fwd Pkts</b></span>, <span style="color:green"><b>Tot Bwd Pkts</b></span>. Therefore, during learning, we will give greater weight to these features to improve model performance.

## Create Training Set and Test set
Now we randomly split the dataset. As we know, the training set is used to train the models, the validation set is used to evaluate the model performance during development, and the test set is used to assess the final performance on unseen data. The random_state parameter, as the name suggests, is used to initialize the internal random number generator, which determines how the data is split into training, validation, and test sets. According to the documentation, setting a random_state ensures that the split is reproducible.

In [ ]:
def assign_test(group):
    n = int(len(group) * 0.2)
    if n > 0:
        idx = np.random.choice(group.index, size=n, replace=False)
        print(idx)
        group.loc[idx, 'isTest'] = 1
    return group

def stratified_train_test_split(dataset):
    dataset["isTest"] = 0
    np.random.seed(7)
    dataset = dataset.groupby("Label" ,group_keys=False).apply(assign_test, meta=dataset)
    train_set = dataset[dataset["isTest"] == 0]
    test_set = dataset[dataset["isTest"] == 1]
    dataset = dataset.drop("isTest", axis=1)
    train_set = train_set.drop("isTest", axis=1)
    test_set = test_set.drop("isTest", axis=1)
    return train_set, test_set

def saveData(train_set, test_set):
    train_set.to_csv("Dataset/train.csv", index=False, single_file=True)
    test_set.to_csv("Dataset/test.csv", index=False, single_file=True) 

train_set, test_set = stratified_train_test_split(dataset)
saveData(train_set, test_set)    